# HireMind — M8 hybrid (E5), seeds 43–45, in priority order (Colab GPU)

Runs the M8 runs one after another, **most promising first**, and saves each one to *My Drive/HireMind_colab_runs* as soon as it finishes (running the notebook again skips finished runs).

| Order | Version | Why |
|---|---|---|
| 1–3 | `minus_struct` (DistilBERT + evidence + Knowledge v1) | best earlier result (seed 42: macro-F1 49.02, Good Fit recall 48.79) |
| 4–6 | `full` (+ structured features) | the plan's proposed model |
| 7–9 | `minus_knowledge` | ablation: is the Knowledge line needed? |
| 10–12 | `minus_evidence` | ablation: is evidence selection needed? |

About 25–40 min per run on a T4 GPU. Same code and settings as the GitHub M8 runs (512 tokens, lr 3e-5, 3 epochs, best-validation-macro-F1 epoch, D0, validation only). Outputs hold scores and row IDs only.

**Two accounts?** Put `RUNS = PRIORITY[:6]` in the first and `RUNS = PRIORITY[6:]` in the second.

**GPU ≠ CPU:** report these as "Colab GPU"; the seed 42 result is a CPU run.


In [ ]:
# 1) Runtime check
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is very slow for DistilBERT: choose Runtime > Change runtime type > GPU)")

In [ ]:
# 2) What to run (in this order)
PRIORITY = [(v, s) for v in ["minus_struct", "full", "minus_knowledge", "minus_evidence"] for s in [43, 44, 45]]
RUNS = PRIORITY            # one account: all 12; two accounts: PRIORITY[:6] and PRIORITY[6:]
SAVE_WEIGHTS = False       # True also keeps each ~260 MB checkpoint
DRIVE_DIR = "/content/drive/MyDrive/HireMind_colab_runs"
print(len(RUNS), "runs:", RUNS)


In [ ]:
# 3) Install the pinned packages (Colab's own CUDA torch is kept)
!pip -q install transformers==4.51.3 pandas==2.2.3 scikit-learn==1.6.1 huggingface-hub==0.30.2 scipy

In [ ]:
# 4) Pinned code: team data loader, team Knowledge v1 code, and the exact experiment scripts used on GitHub
%cd /content
!rm -rf _dl _s2 _m5 _m8 _plan _plan _plan
!git clone -q https://github.com/Hams770/HireMind-DL.git _dl && git -C _dl checkout -q 1f22f34462c400f884bb0a00f061dbea0bd687d5
!git clone -q https://github.com/Hams770/Senior-2.git _s2 && git -C _s2 checkout -q ec28dfc85e618a737cb192cbf959714cfee89aa5
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && cp -r _plan/code/m5-distilbert-plan _m5 && cp _plan/A5_label_overlay_D1/train_labels_v1.csv _m5/phase_c2_m5/e2mods/
!(test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && mkdir -p _m8 && cp -r _plan/code/m8-distilbert/m8 _m8/ && cp _plan/code/m8-seeds/m8/m8_distilbert.py _m8/m8/
!echo "001212500c11a2d471845272718bda0e458968293d551d4979d6331226c178c5  _m5/phase_c2_m5/e2mods/train_labels_v1.csv" | sha256sum -c

In [ ]:
# 5) GPU option: the scripts are CPU-only, so make a patched copy that moves the model and tensors to the GPU.
#    Nothing else changes (same data, same order of batches, same rule for the best epoch).
from pathlib import Path


def patch(src, dst, edits):
    s = Path(src).read_text()
    for a, b in edits:
        assert s.count(a) == 1, ("patch anchor not found exactly once", a[:60])
        s = s.replace(a, b)
    Path(dst).write_text(s)


DEV_LINE = 'DEV = "cuda" if torch.cuda.is_available() else "cpu"; print("device", DEV, flush=True)\n'
patch("/content/_m5/phase_c2_m5/c_stage2_m5_distilbert.py", "/content/_m5/phase_c2_m5/m5_colab.py", [
    ("Vi, Vm = enc(va, L)\n", "Vi, Vm = enc(va, L)\n" + DEV_LINE +
     "model.to(DEV); Xi, Xm, Vi, Vm, ytr = Xi.to(DEV), Xm.to(DEV), Vi.to(DEV), Vm.to(DEV), ytr.to(DEV)\n"),
    ("    p = torch.cat(preds).numpy()\n", "    p = torch.cat(preds).cpu().numpy()\n"),
    ("best_state = {k: v.detach().clone() for", "best_state = {k: v.detach().cpu().clone() for"),
])
patch("/content/_m8/m8/m8_distilbert.py", "/content/_m8/m8/m8_colab.py", [
    ("model = new_model(NF)\n", "model = new_model(NF)\n" + DEV_LINE +
     "model.to(DEV); Xi, Xm, Vi, Vm, ytr = Xi.to(DEV), Xm.to(DEV), Vi.to(DEV), Vm.to(DEV), ytr.to(DEV)\n"
     "Xf = Xf.to(DEV) if Xf is not None else None; Vf = Vf.to(DEV) if Vf is not None else None\n"),
    ("p = torch.cat(preds).numpy(); pr_ = torch.cat(probs).numpy()", "p = torch.cat(preds).cpu().numpy(); pr_ = torch.cat(probs).cpu().numpy()"),
    ("best_state = {k: v.detach().clone() for", "best_state = {k: v.detach().cpu().clone() for"),
    ('f"M8_distilbert_{variant}_s42_best.pt"', 'f"M8_distilbert_{variant}_s{SEED}_best.pt"'),
    ('f"val_predictions_M8_{variant}_s42.csv"', 'f"val_predictions_M8_{variant}_s{SEED}.csv"'),
    ('f"M8_{variant}_s42.json"', 'f"M8_{variant}_s{SEED}.json"'),
])
print("patched copies ready")

In [ ]:
# 6) Run all M8 runs (finished ones in Drive are skipped)
import glob, json, os, shutil, subprocess, sys
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
rows = []
for variant, seed in RUNS:
    if True:
        name = f"m8_{variant}_s{seed}"
        done = f"{DRIVE_DIR}/{name}/summary_row.csv"
        if os.path.exists(done):
            print("skip (already in Drive):", name); rows.append(pd.read_csv(done).iloc[0].to_dict()); continue
        OUT = f"/content/out/{name}"; os.makedirs(OUT, exist_ok=True)
        print("\n=== running", name, flush=True)
        env = dict(os.environ, M8_SEED=str(seed))
        p = subprocess.Popen([sys.executable, "/content/_m8/m8/m8_colab.py", "/content/_dl", OUT, variant, "512"], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content")
        with open(f"{OUT}/full_log.txt", "w") as log:
            for line in p.stdout:
                log.write(line)
                if any(k in line for k in ("device", "max_len", "epoch", "Error", "Traceback", "assert", "wall-time")): print(line, end="")
        if p.wait() != 0:
            print("FAILED:", name, "- see", f"{OUT}/full_log.txt"); shutil.copytree(OUT, f"{DRIVE_DIR}/{name}_FAILED", dirs_exist_ok=True); continue
        if not SAVE_WEIGHTS:
            for f in glob.glob(f"{OUT}/*.pt"): os.remove(f)
        r = json.load(open(glob.glob(f"{OUT}/M8_*_s{seed}.json")[0]))
        row = dict(experiment=f"m8_{variant}", variant=variant, seed=seed, device="colab-" + ("gpu" if torch.cuda.is_available() else "cpu"),
                   accuracy=r["accuracy"], macro_f1=r["macro_f1"], good_fit_recall=r["good_fit_recall"], good_fit_precision=r["good_fit_precision"],
                   potential_fit_recall=r["potential_fit_recall"], best_epoch=r["best_epoch"], stopped_early=r["stopped_early_by_wall_guard"])
        pd.DataFrame([row]).to_csv(f"{OUT}/summary_row.csv", index=False)
        shutil.copytree(OUT, f"{DRIVE_DIR}/{name}", dirs_exist_ok=True)
        print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()}, "-> saved to Drive", flush=True)
        rows.append(row)

In [ ]:
# 7) Averages per version (mean ± SD over the seeds run here) next to the BERT-Tiny baseline
df = pd.DataFrame(rows)
df.to_csv(f"{DRIVE_DIR}/M8_colab_all_runs.csv", index=False)
mets = ["accuracy", "macro_f1", "good_fit_recall", "good_fit_precision", "potential_fit_recall"]
summ = df.groupby("variant")[mets].agg(["mean", "std"]).round(2)
summ["n_seeds"] = df.groupby("variant").seed.nunique()
display(summ)
summ.to_csv(f"{DRIVE_DIR}/M8_colab_summary.csv")
print("BERT-Tiny + Knowledge (M4), 8 CPU seeds: accuracy 56.79, macro-F1 48.61, Good Fit recall 36.72, GF precision 43.68, Potential recall 29.68")
print("Selection rule: beat M4 on accuracy AND Good Fit recall, with no drop > 2 points in GF precision, macro-F1 or Potential recall.")